# 타입 가드와 제네릭 인메모리 저장소

AI 보조 작성 코드입니다. 아래 결과는 실행 스크립트 또는 VS Code의 실제 TypeScript 커널 실행으로 저장합니다.


## 1. 독립 실행을 위한 모델
첫 번째 노트북을 실행하지 않아도 이 노트북 하나로 실행됩니다.


In [1]:
interface Identifiable { id: number; }
type UserRole = 'admin' | 'customer' | 'guest';
interface User extends Identifiable { name: string; email: string; role: UserRole; isActive: boolean; }
type Category = 'electronics' | 'clothing' | 'books';
interface Product extends Identifiable { title: string; price: number; category: Category; stock: number; discountRate?: number; }
const users: User[] = [
 {id:1,name:'Kim',email:'kim@example.com',role:'admin',isActive:true},
 {id:2,name:'Lee',email:'lee@example.com',role:'customer',isActive:true},
 {id:3,name:'Park',email:'park@example.com',role:'guest',isActive:false}
];
const products: Product[] = [
 {id:1,title:'Slacks',price:40000,category:'clothing',stock:10,discountRate:0.1},
 {id:2,title:'Book',price:20000,category:'books',stock:4},
 {id:3,title:'Keyboard',price:50000,category:'electronics',stock:2,discountRate:0}
];
console.log('회원',users.length,'상품',products.length);


회원 3 상품 3


## 2. 타입 좁히기와 저장소
`'email' in item`이 참이면 User, 거짓이면 Product입니다. `T extends Identifiable`은 공통 id만 요구하며 저장소별 실제 필드는 유지합니다. 원문 `<T Identifiable extends>`는 문법 오타이므로 `<T extends Identifiable>`로 바로잡았습니다.


In [2]:
function renderItemSummary(item: User | Product): string {
 if ('email' in item) return '[회원] '+item.name+' ('+item.role+')';
 return '[상품] '+item.title+' - '+item.price+'원';
}
class InMemoryStorage<T extends Identifiable> {
 private items: T[] = [];
 add(item: T): void {
  if(this.items.some(existing=>existing.id===item.id)) throw new Error('중복 ID');
  this.items.push({...item});
 }
 getById(id: number): T | undefined { const found=this.items.find(item=>item.id===id); return found ? {...found}:undefined; }
 getAll(): T[] { return this.items.map(item=>({...item})); }
 update(item: T): void {
  if(!this.items.some(existing=>existing.id===item.id)) throw new Error('없는 ID');
  this.items=this.items.map(existing=>existing.id===item.id?{...item}:existing);
 }
 delete(id: number): boolean { const before=this.items.length;this.items=this.items.filter(item=>item.id!==id);return this.items.length<before; }
}
const userStorage=new InMemoryStorage<User>();
const productStorage=new InMemoryStorage<Product>();
users.forEach(user=>userStorage.add(user));products.forEach(product=>productStorage.add(product));
console.log(userStorage.getById(1));console.log(productStorage.getAll());
console.log(renderItemSummary(users[0]));console.log(renderItemSummary(products[0]));
if(userStorage.getById(999)!==undefined) throw new Error('미존재 ID 오류');
const copy=userStorage.getAll();copy[0].name='changed';
if(userStorage.getById(1)?.name==='changed') throw new Error('저장소 참조 노출');
try {userStorage.add(users[0]);throw new Error('중복 검증 실패');} catch(e: unknown) {if(!(e instanceof Error)||e.message!=='중복 ID')throw e;}
console.log('타입 가드 양 분기, 미존재 ID, 복사 반환, 중복 검증 통과');


{
  id: 1,
  name: 'Kim',
  email: 'kim@example.com',
  role: 'admin',
  isActive: true
}


[
  {
    id: 1,
    title: 'Slacks',
    price: 40000,
    category: 'clothing',
    stock: 10,
    discountRate: 0.1
  },
  { id: 2, title: 'Book', price: 20000, category: 'books', stock: 4 },
  {
    id: 3,
    title: 'Keyboard',
    price: 50000,
    category: 'electronics',
    stock: 2,
    discountRate: 0
  }
]


[회원] Kim (admin)


[상품] Slacks - 40000원


타입 가드 양 분기, 미존재 ID, 복사 반환, 중복 검증 통과


## 3. 동작과 캡슐화 검증
반환 배열뿐 아니라 현재 모델의 객체도 얕게 복사합니다. User/Product는 원시값만 가진 평면 구조이므로 충분합니다. 중첩 객체를 추가한다면 깊은 복사 정책이 별도로 필요합니다.


In [3]:
// 추가 시 복사, 조회 시 복사, 수정과 삭제까지 같은 제네릭 클래스를 재사용합니다.
const added: User = { id: 99, name: 'Original', email: 'test@example.com', role: 'customer', isActive: true };
userStorage.add(added);
added.name = 'Outside change';
if (userStorage.getById(99)?.name !== 'Original') throw new Error('add 참조 노출');
const single = userStorage.getById(99);
if (!single) throw new Error('조회 실패');
single.name = 'Changed copy';
if (userStorage.getById(99)?.name !== 'Original') throw new Error('getById 참조 노출');
userStorage.update({ ...single, name: 'Updated' });
if (userStorage.getById(99)?.name !== 'Updated') throw new Error('수정 실패');
if (!userStorage.delete(99) || userStorage.delete(99)) throw new Error('삭제 결과 오류');
if (userStorage.getById(99) !== undefined) throw new Error('삭제 후 조회 오류');
let missingRejected = false;
try { userStorage.update(added); } catch (error: unknown) { missingRejected = error instanceof Error && error.message === '없는 ID'; }
if (!missingRejected) throw new Error('없는 항목 수정이 허용됨');
const summaries: string[] = [...users, ...products].map(renderItemSummary);
console.log(summaries.join('\n'));
console.log('PASS: 추가·단건조회 복사 보호 / 수정 / 삭제 / 없는 항목 수정 거절 / 모든 User·Product 분기');


[회원] Kim (admin)
[회원] Lee (customer)
[회원] Park (guest)
[상품] Slacks - 40000원
[상품] Book - 20000원
[상품] Keyboard - 50000원


PASS: 추가·단건조회 복사 보호 / 수정 / 삭제 / 없는 항목 수정 거절 / 모든 User·Product 분기
